# Cross-Validation

In [1]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
sns.set_style('whitegrid')
import pandas as pd


In [2]:
from sklearn.model_selection import train_test_split
from sklearn import datasets
from sklearn import svm

In [3]:
from sklearn.datasets import fetch_openml

boston = fetch_openml(name="boston", version=1, as_frame=True)
data = boston.frame
df= data.drop('MEDV', axis=1)
y=boston.target

In [4]:
df.shape, y.shape

((506, 13), (506,))

we can now quickly sample a training set while holding out 40% of the data for testing(evaluating) our regressor:

In [5]:
X_train,X_test,y_train,y_test=train_test_split(df,y,test_size=0.4,random_state=0)
print(X_train.shape,y_train.shape)
print(X_test.shape,y_test.shape)
regression=svm.SVR(kernel='linear',C=1).fit(X_train,y_train)
regression.score(X_test,y_test)

(303, 13) (303,)
(203, 13) (203,)


0.6674313821662665

---

## Computing Cross-Validated metrics

In [6]:
from sklearn.model_selection import cross_val_score
regression=svm.SVR(kernel='linear',C=1)
scores=cross_val_score(regression,df,y,cv=5)
scores

array([0.77275763, 0.72778244, 0.56131914, 0.1505652 , 0.08212413])

The mean score and the 95% confidence interval of the score estimate are hence given by:

In [7]:
print("Accuracy: %0.2f(+/- %0.2f)" %(scores.mean(),scores.std()))

Accuracy: 0.46(+/- 0.29)


By dfault,the score computed at each CV iteration is the score method of the estimator.it is possible to charge this by using the scoring paramter.

In [8]:
from sklearn import metrics

In [9]:
scores=cross_val_score(regression,df,y,cv=5,scoring='neg_mean_squared_error')
scores

array([ -7.8478596 , -24.78180305, -35.13272325, -74.50549945,
       -24.40477437])

---

## K-fold

In [10]:
import numpy as np
from sklearn.model_selection import KFold

X=["a,","b","c","d"]
kf=KFold(n_splits=2)
for train, test in kf.split(X):
    print("%s %s" %(train,test))

[2 3] [0 1]
[0 1] [2 3]


---

## Stratified K-fold

In [11]:
from sklearn.model_selection import StratifiedKFold

X=np.ones(10)
y=[0,0,0,0,1,1,1,1,1,1]
skf=StratifiedKFold(n_splits=3)
for train, test in skf.split(X,y):
    print("%s %s" %(train,test))

[2 3 6 7 8 9] [0 1 4 5]
[0 1 3 4 5 8 9] [2 6 7]
[0 1 2 4 5 6 7] [3 8 9]


In [12]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
# from sklearn.linear_model import LogicalRegression
from sklearn import svm
from sklearn.pipeline import make_pipeline

# pipe_lr=make_pipeline(StandardScaler(),
#                    PCA(n_components=2),
#                    LogicalRegression(random_state=1))
pipe_svm=make_pipeline(StandardScaler(),
                   PCA(n_components=2),
                   svm.SVR(kernel='linear',C=1))
pipe_svm.fit(X_train,y_train)
y_pred=pipe_svm.predict(X_test)
print("Test Accuracy: %.3f" %pipe_svm.score(X_test,y_test))

Test Accuracy: 0.391


In [13]:
from sklearn.model_selection import cross_val_score
scores=cross_val_score(estimator=pipe_svm,
                       X=X_train,
                       y=y_train,
                       cv=10,
                       n_jobs=1)
print("CV accuracy scores:  %s" %scores)

CV accuracy scores:  [0.63971176 0.43579197 0.46977821 0.25027246 0.5124364  0.26221374
 0.30877195 0.54528563 0.37810066 0.47313549]


In [14]:
print("CV accuracy: %.3f +/- %.3f" %(np.mean(scores),
                               np.std(scores)))

CV accuracy: 0.428 +/- 0.121
